Submit a .py script that: (1) fetches the MORTGAGE30US series directly from FRED, (2) resamples it to
monthly averages, (3) merges it onto both the combined sold and listings datasets using a year_month
key, and (4) includes a validation check confirming no null rate values exist after the merge. Save both
enriched datasets as new CSVs.

In [1]:
import pandas as pd

In [14]:
# Step 1 – Fetch the mortgage rate data from FRED

print("Fetching Mortgage Data...\n")
url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=MORTGAGE30US"
mortgage = pd.read_csv(url, parse_dates=['observation_date'])
mortgage.columns = ['date', 'rate_30yr_fixed']

print("Creating monthly averages...\n")
# Step 2 – Resample weekly rates to monthly averages
mortgage['year_month'] = mortgage['date'].dt.to_period('M')
mortgage_monthly = (
mortgage.groupby('year_month')['rate_30yr_fixed']
.mean()
.reset_index()
)

sold = pd.read_csv("data/sold_clean.csv", low_memory=False)
listings = pd.read_csv("data/listing_clean.csv", low_memory=False)

print("Creating year_month keys...\n")
# Step 3 – Create a matching year_month key on the MLS datasets
# Sold dataset — key off CloseDate
sold['year_month'] = pd.to_datetime(sold['CloseDate']).dt.to_period('M')
# Listings dataset — key off ListingContractDate
listings['year_month'] = pd.to_datetime(
listings['ListingContractDate']
).dt.to_period('M')

print("Merging on year_month...\n")
# Step 4 – Merge
sold_with_rates = sold.merge(mortgage_monthly, on='year_month', how='left')
listings_with_rates = listings.merge(mortgage_monthly, on='year_month', how='left')

print("Checking Null Rate:")
# Step 5 – Validate the merge
# Check for any unmatched rows (rate should not be null)
print("SOLD Null rate of 'rate_30yr_fixed':", sold_with_rates['rate_30yr_fixed'].isnull().sum())
print("LISTING Null rate of 'rate_30yr_fixed':", listings_with_rates['rate_30yr_fixed'].isnull().sum())

listings_with_rates.to_csv("listing_enriched.csv", index=False)

sold_with_rates.to_csv("sold_enriched.csv", index=False)

print("\nCSVs Created!")

Fetching Mortgage Data...

Creating monthly averages...

Creating year_month keys...

Merging on year_month...

Checking Null Rate:
SOLD Null rate of 'rate_30yr_fixed:' 0
LISTING Null rate of 'rate_30yr_fixed:' 0

CSVs Created!
